FastAPI → Request/response validation.

Configuration Management → Validate environment variables or config files.

Data Pipelines → Ensure incoming data matches schema.

ORM Integration → Works with SQLAlchemy, Django, and SQLModel.

Testing → Validate mock data quickly.

Field validators → validate single fields.

Model validators → validate relationships across fields.

Nested models → validation applies recursively.

Errors → structured, easy to debug or return in A

In [ ]:
#defining basic model using pydantic
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: int
    email: str

user = UserProfile(name="Liam Svensson", age=40, email="liam.svensson@example.com")
print(user)

In [ ]:
#working with model attributes
print(user.name)
print(user.age)
print(user.email)

In [ ]:
#handling default values and required fields
class UserProfileWithDefaults(BaseModel):
    name: str
    age: int = 30  # default value
    email: str  

user_with_defaults = UserProfileWithDefaults(name="Emma Johansson", email="venkat@gmail.com")
print(user_with_defaults)

In [ ]:
#Advanced Validation Techniques - with decorators and custome validators
from pydantic import BaseModel, field_validator, EmailStr
class UserProfileWithValidation(BaseModel):
    name: str
    age: int
    email: EmailStr  # using EmailStr for email validation

    @field_validator('age')
    def validate_age(cls, value):
        if value < 0:
            raise ValueError("Age must be a non-negative integer")
        return value

# Valid
user_with_validation = UserProfileWithValidation(name="Olivia Eriksson", age=25, email="olivia.eriksson@example.com")
print(user_with_validation)

# Invalid
try:
    bad_user = UserProfileWithValidation(name="Noah Lindqvist", age=-5, email="noah.lindqvist@example.com")
except Exception as e:
    print("Validation failed:", e)

In [ ]:
#model validators
from pydantic import BaseModel, model_validator, EmailStr
class UserProfileWithModelValidator(BaseModel):
    name: str
    age: int
    email: EmailStr  # using EmailStr for email validation

    @model_validator(mode='before')
    def check_age(cls, values):
        age = values.get('age')
        if age is not None and age < 0:
            raise ValueError("Age must be a non-negative integer")
        return values

In [ ]:
from pydantic import BaseModel, EmailStr

class User(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode='before') #mode = before,after,wrap
    def validate_passwords(cls, values):
        password = values.get('password')
        confirm_password = values.get('confirm_password')
        if password != confirm_password:
            raise ValueError("Passwords do not match")
        return values
    
user = User(password="secure123", confirm_password="secure123")
print(user)


In [30]:
#password or masking the sensitive data
from pydantic import BaseModel, SecretStr, model_validator

class User(BaseModel):
    password: SecretStr
    confirm_password: SecretStr

    @model_validator(mode="before")
    def validate_passwords(cls, values):
        if values.get("password") != values.get("confirm_password"):
            raise ValueError("Passwords do not match")
        return values

user = User(password="secure123", confirm_password="secure123")
print(user)   # password=SecretStr('**********') confirm_password=SecretStr('**********')

# Access actual value if needed
print(user.password.get_secret_value())  # secure123


password=SecretStr('**********') confirm_password=SecretStr('**********')
secure123


In [ ]:
from pydantic import BaseModel, EmailStr
class UserProfile(BaseModel):
    name: str
    age: int
    email: EmailStr

data = '{"name": "Noah Kim", "age": 28, "email": "noah.kim@example.kr"}'
# parsing data from JSON string to Pydantic model
uf = UserProfile.model_validate_json(data)
print(uf)

#serialize the model back to JSON
user = UserProfile(name="Luca Rossi", age=29, email="luca.rossi@example.it")
json_data = user.model_dump_json()
print(json_data)

In [ ]:
#Handling Optional and Nullable Fields
#Pydantic allows optional fields using Python’s Optional type which default to None if not provided. This is useful for fields that may be missing.
from typing import Optional
from pydantic import BaseModel

class UserProfile(BaseModel):
    name: str
    age: Optional[int] = None
    email: str

user = UserProfile(name="Mia Johansson", email="mia.johansson@example.se")
print(user)

In [ ]:
#custom field validation 
from pydantic import BaseModel, field_validator
class Product(BaseModel):
    name: str
    price: float

    @field_validator('price')
    def price_must_be_positive(cls, v):
        if v <= 5000:
            raise ValueError('Price must be positive')
        return v

product = Product(name="Laptop", price=6000)
print(product)

In [ ]:
#Nested Models + Serialization
from pydantic import BaseModel
from typing import List

class Item(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    id: int
    items: List[Item]

order = Order(id=1, items=[Item(name="Laptop", price=6000), Item(name="Mouse", price=50)])
print(order)

In [ ]:
#cross field validation 
from pydantic import BaseModel, model_validator

class Booking(BaseModel):
    start_date: str
    end_date: str

    @model_validator(mode="after")
    def check_dates(self):
        if self.start_date >= self.end_date:
            raise ValueError("End date must be after start date")
        return self

# Valid
booking = Booking(start_date="2026-10-08", end_date="2026-10-10")
print(booking)

# Invalid
try:
    bad_booking = Booking(start_date="2026-10-10", end_date="2026-10-08")
except Exception as e:
    print("Validation failed:", e)


Pydantic Settings Management: 
  - Pydantic provides BaseSettings to manage configuration via environment variables, .env files and other sources. This is useful for real-world apps where settings may change between environments.

  - class Settings(BaseSettings): Defines a settings model that reads environment variables automatically.
  - debug: bool = False: Optional field with a default value.
  - class Config: env_file = ".env": Instructs Pydantic to also load values from a .env file.
  - Settings(...): Instantiates the settings, validating all fields.

In [ ]:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str
    debug: bool = False
    database_url: str

    class Config:
        env_file = ".env"

settings = Settings(app_name="TestApp", database_url="sqlite:///:memory:")
print(settings.app_name)

Aliasing and Field Customization
    - Pydantic allows you to set aliases, default values and metadata for fields. This is useful when your external data uses different field names or when you want to provide descriptive information for documentation.
    
    full_name: str = Field(..., alias="name"): Maps incoming name to internal full_name.
    age: int = Field(18, description="Age of the user"): Default value of 18 with metadata description.
    User(name="Rita"): Instantiates the model using the alias.
    user.model_dump(): Returns the model as a dictionary with internal field names.

In [29]:
from pydantic import BaseModel, Field
class AliasedUserProfile(BaseModel):
    name: str = Field(..., alias="full_name",title="full_name", description="The user's full name")
    age: int = Field(..., alias="age",ge=0, le=120, description="The user's age (0-120)")
    email:Optional[str] = Field(None, alias="email_address", description="The user's email address")

#pass the values using the alias names
aliased_user = AliasedUserProfile(full_name="Liam Svensson", age=40, email_address="venkat@gmail.com")
print(aliased_user)

name='Liam Svensson' age=40 email='venkat@gmail.com'
